## Importaciones y Configuración

* `pathlib.Path:` Permite trabajar de manera flexible y clara con rutas de archivos y directorios.

* `np y pd:` Importaciones habituales de NumPy y pandas para manejar arreglos y tablas de datos.

* `DATA_DIR:` Se define la ruta a la carpeta datos ubicada un nivel arriba del directorio actual (parent). Path.cwd().resolve() obtiene la ruta absoluta de trabajo.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

## Carga del Dataset

In [3]:
# En caso de que se ejecute clonando el repositorio

# DATA_DIR = Path.cwd().resolve().parent / "datos"

# datos_titanic = pd.read_parquet(DATA_DIR / "02_datos_con_tipo_de_dato_ajustado_titanic.parquet", engine="pyarrow")

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# OJO: La ruta de los archivos dependerá de cada Drive
ruta_de_archivos = r"/content/drive/MyDrive/cursos-para-dictar/UDM/06_clase/titanic/datos/02_datos_con_tipo_de_dato_ajustado_titanic.parquet"

datos_titanic = pd.read_parquet(ruta_de_archivos)

## Selección de Columnas y Estructura del DataFrame

In [6]:
columnas_seleccionadas = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
    "survived",
]

In [7]:
df_titanic = datos_titanic[columnas_seleccionadas]

df_titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   pclass    1309 non-null   int64   
 1   sex       1309 non-null   category
 2   age       1046 non-null   float64 
 3   sibsp     1309 non-null   int8    
 4   parch     1309 non-null   int8    
 5   fare      1308 non-null   float64 
 6   embarked  1307 non-null   category
 7   survived  1309 non-null   bool    
dtypes: bool(1), category(2), float64(2), int64(1), int8(2)
memory usage: 37.5 KB


## Ajuste de Tipos y Eliminación de Duplicados

In [8]:
df_titanic["survived"] = df_titanic["survived"].astype(bool).astype(int)

<ipython-input-8-dfda65542c4f>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_titanic["survived"] = df_titanic["survived"].astype(bool).astype(int)


In [9]:
df_titanic = df_titanic.drop_duplicates()

# AutoML


A continuación, se muestra un ejemplo **completamente documentado** sobre cómo usar [PyCaret](https://pycaret.gitbook.io/docs#classification) para llevar a cabo un proceso de **preprocesamiento y selección de modelos** de manera similar a lo que hace un flujo de trabajo con *ColumnTransformer* y *Pipeline* en scikit-learn.

## ¿Por Qué No Se Usa Pipeline ni ColumnTransformer Directamente?

PyCaret **automatiza** todo el proceso de:
- Imputación de valores nulos  
- Codificación de variables categóricas (One-Hot o Label Encoding)  
- Opciones de escalado/normalización (si lo solicitas)  
- Selección y entrenamiento de múltiples algoritmos  

Internamente, `PyCaret` construye su propio pipeline con estos pasos – no tienes que definirlo manualmente. Por esta razón, **no** se ven explícitos `ColumnTransformer` ni `Pipeline`, como lo harías en código puro de scikit-learn. `PyCaret`, al llamar a la función `setup`, crea un pipeline que incluye las transformaciones de preprocesamiento y lo aplica automáticamente a todos los modelos que compara o entrena.

In [ ]:
# Esta instalación puede tardar unos 2 minutos

!pip install pycaret

Normalmente, luego de instalar `PyCaret` es necesario **reiniciar el runtime** debido aque se instalan dependencias específicas de `pandas`, `scikit-learn`, `matplotlib`.

Si estás usando **GOOGLE COLAB** simplemente debes de ir a **"Entorno de ejecución"** --> **"Reiniciar entorno de ejecución"**.

In [15]:
# Importar las funciones principales de clasificación en PyCaret

from pycaret.classification import setup, compare_models

In [16]:
# Definir la columna objetivo
target_col = "survived"

Para leer más de la documentación oficial del método `setup` dirigirse a:

https://pycaret.readthedocs.io/en/stable/api/classification.html

In [19]:
# Llamada a setup
clf_setup = setup(
    data=df_titanic,          # DataFrame con TODAS las columnas
    target=target_col,        # Nombre de la columna objetivo
    session_id=42,            # Semilla para reproducibilidad (opcional pero recomendable)

    # Parámetros para controlar el preprocesamiento:
    numeric_features=["age", "fare", "sibsp", "parch"],  # Columnas que consideramos numéricas
    categorical_features=["sex", "embarked"],            # Columnas categóricas sin orden
    ordinal_features={"pclass": [1, 2, 3]},              # pclass es una variable categórica con orden

    # Estrategias de imputación
    numeric_imputation="median",  # Reemplaza valores nulos numéricos con la mediana
    categorical_imputation="mode", # Reemplaza valores nulos categóricos con el valor más frecuente

    # Con esto PyCaret creará automáticamente un pipeline que hace:
    # 1) Imputación de NaN en columnas numéricas con 'median'
    # 2) Imputación de NaN en columnas categóricas con 'mode'
    # 3) Encoding ordinal para pclass
    # 4) Encoding one-hot o label para otras columnas categóricas

)

,Description,Value
0,Session id,42
1,Target,survived
2,Target type,Binary
3,Original data shape,"(1114, 8)"
4,Transformed data shape,"(1114, 10)"
5,Transformed train set shape,"(779, 10)"
6,Transformed test set shape,"(335, 10)"
7,Ordinal features,1
8,Numeric features,4
9,Categorical features,2


En `PyCaret`, no necesitas separar manualmente los datos en conjuntos de entrenamiento y prueba. Esto se hace automáticamente al llamar a la función `setup`. A continuación, se explica el proceso y cómo ajustar (fine-tune) el modelo GBC.

## Comparación de varios modelos


La forma más sencilla de probar múltiples algoritmos de clasificación es llamar a compare_models. Por defecto, probará un conjunto amplio (más de 10) de algoritmos. Podemos restringirlo con el parámetro `include`.

In [20]:
best_model = compare_models(
    include=["lr", "rf", "gbc", "lightgbm"],  # Solo estos 4 algoritmos
    sort="Accuracy",   # Métrica principal para comparar
    n_select=4         # Selecciona y devuelve los 4 mejores
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.7894,0.8319,0.6265,0.8386,0.7103,0.5516,0.5709,0.1820
lightgbm,Light Gradient Boosting Machine,0.7651,0.8215,0.6480,0.7641,0.6974,0.5078,0.5157,0.4230
lr,Logistic Regression,0.7535,0.8193,0.6689,0.7274,0.6907,0.4873,0.4935,0.8600
rf,Random Forest Classifier,0.7394,0.7991,0.6385,0.7147,0.6685,0.4560,0.4627,0.3080


Processing:   0%|          | 0/24 [00:00<?, ?it/s]

En una inspección rápida, el mejor modelo fue el **Gradient Boosting Classifier**, con la mayor métrica en `Accuracy`. Por lo tanto, exploremos este modelo un poco más y ajsutémoslo.

In [23]:
from pycaret.classification import create_model, tune_model, predict_model

# 1. Crear el modelo GBC por defecto
print("Creando modelo GBC...")
gbc_model = create_model("gbc")

# 2. Afinar hiperparámetros
print("Realizando fine-tuning del modelo GBC...")
gbc_tuned = tune_model(gbc_model)

# 3. Mostrar el resumen del modelo ajustado
print("Resumen del modelo ajustado:")
print(gbc_tuned)  # Esto imprimirá un resumen con los hiperparámetros y rendimiento

# 4. Predecir sobre la partición interna de PyCaret
print("Generando predicciones en el conjunto de validación interno:")
predicciones_internas = predict_model(gbc_tuned)

print(predicciones_internas.head())

Creando modelo GBC...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7821,0.8589,0.5455,0.9000,0.6792,0.5288,0.5669
1,0.7692,0.8266,0.7273,0.7273,0.7273,0.5273,0.5273
2,0.7949,0.7737,0.6364,0.8400,0.7241,0.5658,0.5796
3,0.7821,0.8421,0.5758,0.8636,0.6909,0.5328,0.5589
4,0.7821,0.8795,0.6667,0.7857,0.7213,0.5443,0.5493
5,0.8077,0.8391,0.6667,0.8462,0.7458,0.5946,0.6055
6,0.8462,0.8498,0.6970,0.9200,0.7931,0.6743,0.6908
7,0.8205,0.9110,0.7812,0.7812,0.7812,0.6291,0.6291
8,0.7436,0.7463,0.4688,0.8333,0.6000,0.4323,0.4711


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

Realizando fine-tuning del modelo GBC...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7821,0.8360,0.5152,0.9444,0.6667,0.5247,0.5780
1,0.8077,0.8337,0.6970,0.8214,0.7541,0.5979,0.6034
2,0.7692,0.8232,0.5455,0.8571,0.6667,0.5032,0.5333
3,0.7949,0.8242,0.5455,0.9474,0.6923,0.5546,0.6022
4,0.7949,0.8764,0.5758,0.9048,0.7037,0.5584,0.5918
5,0.7692,0.8350,0.5455,0.8571,0.6667,0.5032,0.5333
6,0.8205,0.8791,0.6061,0.9524,0.7407,0.6136,0.6503
7,0.7821,0.8781,0.5938,0.8261,0.6909,0.5295,0.5467
8,0.7308,0.7242,0.4062,0.8667,0.5532,0.3947,0.4528


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
Resumen del modelo ajustado:
GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                           learning_rate=0.1, loss='log_loss', max_depth=3,
                           max_features=None, max_leaf_nodes=None,
                           min_impurity_decrease=0.0, min_samples_leaf=1,
                           min_samples_split=2, min_weight_fraction_leaf=0.0,
                           n_estimators=100, n_iter_no_change=None,
                           random_state=42, subsample=1.0, tol=0.0001,
                           validation_fraction=0.1, verbose=0,
                           warm_start=False)
Generando predicciones en el conjunto de validación interno:


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.7881,0.8732,0.7286,0.7556,0.7418,0.5622,0.5624


      pclass     sex   age  sibsp  parch        fare embarked  survived  \
587        2    male   2.0      1      1   23.000000        S         1   
0          1  female  29.0      0      0  211.337494        S         1   
458        2  female  17.0      0      0   10.500000        S         1   
1238       3    male  25.0      0      0    7.795800        S         1   
797        3    male  40.5      0      0    7.750000        Q         0   

      prediction_label  prediction_score  
587                  1            0.7050  
0                    1            0.9485  
458                  1            0.8306  
1238                 0            0.7266  
797                  0            0.9173  


## Entendiendo la Columna "prediction_score"

La columna **prediction_score** no indica que la predicción sea 100% o 0%; en cambio, representa la **confianza** o **probabilidad** asignada por el modelo a que una muestra pertenezca a una determinada clase.

### ¿Qué es "prediction_score"?

- **Probabilidad de Clase:**  
  La mayoría de los modelos de clasificación generan una probabilidad para cada clase (por ejemplo, la probabilidad de que la muestra pertenezca a la clase 1).  

  - Por ejemplo, un score de 0.7050 significa que el modelo cree que hay un 70.5% de probabilidad de que la muestra sea de la clase 1.
  
- **Umbral de Decisión:**  

  Se establece un umbral (típicamente 0.5) para convertir la probabilidad en una predicción binaria.  

  - Si el score es mayor que 0.5, la predicción se etiqueta como 1.  
  - Si es menor, se etiqueta como 0.
  
- **Calibración y Modelado:**  

  La puntuación rara vez es exactamente 1 o 0 porque los modelos operan con incertidumbre inherente.

  - Un score cercano a 1 indica alta confianza, pero es normal que incluso las predicciones correctas tengan un valor menor a 1.

  - Esto permite evaluar qué tan "seguro" está el modelo de cada predicción, lo que puede ser útil para decisiones posteriores.


### Conclusión

- La **prediction_score** es una medida de **confianza** y no un indicador de corrección absoluta.
- Las predicciones se obtienen al aplicar un umbral (usualmente 0.5) a estos scores.
- Es normal que incluso las predicciones correctas no tengan un score de 1, ya que el modelo siempre maneja un grado de incertidumbre en sus predicciones.


## ¿Cómo obtengo los parámetros del mejor modelo?

Una vez que has ajustado (fine-tuned) el modelo en PyCaret, el objeto devuelto (por ejemplo, `gbc_tuned`) es un estimador de scikit-learn. Para ver su configuración (hiperparámetros) puedes usar el método `get_params()`. Por ejemplo:

In [24]:
# Imprimir los parámetros del mejor modelo ajustado
print(gbc_tuned.get_params())

{'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}


## Probar el modelo en datos no vistos

In [25]:
# Número de muestras sintéticas
np.random.seed(57)

n_samples = 50

# Generar datos sintéticos siguiendo la estructura del dataset Titanic
df_synthetic_test = pd.DataFrame({
    "pclass": np.random.choice([1, 2, 3], size=n_samples),
    "sex": np.random.choice(["male", "female"], size=n_samples),
    "age": np.random.uniform(0, 80, size=n_samples),
    "sibsp": np.random.randint(0, 4, size=n_samples),
    "parch": np.random.randint(0, 4, size=n_samples),
    "fare": np.random.uniform(10, 100, size=n_samples),
    "embarked": np.random.choice(["C", "Q", "S"], size=n_samples)
})

# Mostrar los primeros registros para verificar
df_synthetic_test.head()

,pclass,sex,age,sibsp,parch,fare,embarked
0,3,male,29.549676,2,2,87.460519,C
1,2,male,22.782733,2,1,81.690288,S
2,3,male,74.169372,1,2,51.914214,Q
3,1,male,62.041734,3,2,74.020331,Q
4,3,male,51.305236,0,0,26.458023,S


### 💥❗ Garantizar que a los Datos Sintéticos se les Aplique el Mismo Preprocesamiento :


Cuando configuraste `PyCaret` con la función `setup`, se creó internamente un pipeline que realiza:

* Imputación de valores faltantes (según las estrategias definidas).

* Codificación de variables numéricas, categóricas y ordinales.


Este pipeline se guarda y se aplica automáticamente a cualquier dato nuevo que pases a la función predict_model.

In [26]:
# Aplicar el modelo ajustado a los datos sintéticos de prueba
predicciones_sinteticas = predict_model(gbc_tuned, data=df_synthetic_test)

# Mostrar las predicciones (incluye 'prediction_label' y 'prediction_score')
predicciones_sinteticas.head()

,pclass,sex,age,sibsp,parch,fare,embarked,prediction_label,prediction_score
0,3,male,29.549675,2,2,87.460518,C,0,0.6397
1,2,male,22.782732,2,1,81.690285,S,0,0.8779
2,3,male,74.169373,1,2,51.914215,Q,0,0.9702
3,1,male,62.041733,3,2,74.020332,Q,0,0.8433
4,3,male,51.305237,0,0,26.458023,S,0,0.8716


# Guardar el Modelo

In [27]:
from pycaret.classification import save_model

In [28]:
# Guardar el modelo en un archivo llamado 'gbc_tuned_model.pkl'

save_model(gbc_tuned, model_name='gbc_tuned_model')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['age', 'fare', 'sibsp', 'parch'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='median'))),
                 ('categorical_imputer',
                  TransformerWrapper(exclude=None, include=['sex', 'emba...
                                             criterion='friedman_mse', init=None,
                                             learning_rate=0.1, loss='log_loss',
              

# Cargar el modelo

In [29]:
from pycaret.classification import load_model

# Cargar el modelo previamente guardado
gbc_tuned_loaded = load_model('gbc_tuned_model')

Transformation Pipeline and Model Successfully Loaded


# Exportar en otros formatos

Si prefieres utilizar joblib para guardar el modelo (ya que también es común y a veces más rápido para objetos grandes), puedes hacerlo manualmente, ya que el modelo devuelto por PyCaret es compatible con scikit-learn. Por ejemplo:

In [30]:
import joblib

# Exportar el modelo usando joblib en lugar de pickle
joblib.dump(gbc_tuned, 'gbc_tuned_model.joblib')

# Para cargarlo luego:
gbc_tuned_loaded = joblib.load('gbc_tuned_model.joblib')

# Referencias

* https://pycaret.gitbook.io/docs#classification

* https://pycaret.readthedocs.io/en/stable/api/classification.html

* https://medium.com/@roshmitadey/pycaret-for-autom-a7b5c36a5772

